# 04 · Motor multicampo y evaluacion con metricas — nivel EXPERTO

**Que hace**: deduplica usando **varias variables a la vez** (NIT, razon social,
email, telefono, direccion, ciudad) con pesos y politicas configurables, incluida la
**corroboracion de veto de NIT (Fase 3)**: dos registros con NIT distinto
pueden unirse SOLO si un campo fuerte identico (email/telefono) + nombre similar lo
corroboran. Ademas calcula **precision / recall / F1** contra una verdad de referencia.

**Estructura**:
1. **Demo autocontenida** (datos sinteticos embebidos): corre sin archivos y muestra
   el efecto OFF vs ON de la corroboracion.
2. **Sus datos**: mapee columnas, ajuste pesos/umbrales y ejecute (rutas en la Celda A1).
3. **Evaluacion**: si tiene columna de verdad (`ID_GRUPO` real), obtenga P/R/F1.

> En Colab Free, suba la base a `/content/datos` y descargue `/content/resultados`
> antes de cerrar la sesion; el disco local es efimero.
>
> Motor: `rues-linker 0.22.4` — `EsquemaCampos`, `CorroboracionVeto`,
> `evaluar_esquema`, `clusters_desde_decisiones`, `evaluar_pares`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.22.4"
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"


def _version_ok(v: str, objetivo: str = VERSION_OBJETIVO) -> bool:
    """True si la version instalada coincide con el release objetivo."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) == _t(objetivo)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza exactamente rues-linker 0.22.4 desde el tag oficial."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada):
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub...")
    else:
        print(f"rues-linker {instalada} no coincide con {VERSION_OBJETIVO}; instalando...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada. "
            "Reinicie el entorno y vuelva a ejecutar esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_OBJETIVO}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS  (donde estan sus datos y donde quedan los resultados)
# ══════════════════════════════════════════════════════════════════════════
# En Colab Free suba sus archivos a /content/datos (los notebooks 01-04 no
# montan Drive) y descargue RUTA_RESULTADOS antes de cerrar la sesion: el
# disco local es efimero. Los nombres de archivo se declaran en la Celda B.
# tests/test_notebooks_contrato.py sobreescribe estas rutas para correr el
# notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/datos")  # carpeta con sus archivos
RUTA_RESULTADOS = Path("/content/resultados/04")  # carpeta de salida

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════

# ── Lectura y validacion de bases (fail-fast con mensajes accionables) ──────

import unicodedata
from pathlib import Path

import pandas as pd

from record_linkage import (
    CampoSpec,
    CorroboracionVeto,
    EsquemaCampos,
    PoliticaFaltante,
    TipoCampo,
    clusters_desde_decisiones,
    evaluar_esquema,
    evaluar_pares,
)
from record_linkage.exporters.escritor import LIMITE_FILAS_EXCEL, escribir_csv_gz, escribir_xlsx
from record_linkage.matching.motor_bloqueo import BloqueoComponible, LlaveExacta, LSHTexto


def _norm(texto: str) -> str:
    """Normaliza un nombre de columna: sin tildes, mayusculas, sin espacios extremos."""
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).strip().upper()


def leer_base(
    ruta: str | Path,
    hoja: str | int = 0,
    fila_encabezado: int = 0,
    columnas_texto: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Lee CSV/XLSX validando existencia y devolviendo columnas normalizadas.

    Args:
        ruta: ruta local al archivo .csv, .xlsx o .xls.
        hoja: nombre o indice de hoja (solo Excel).
        fila_encabezado: fila (0-index) donde estan los encabezados.
        columnas_texto: columnas a forzar como texto (p. ej. NIT, para no
            perder ceros a la izquierda ni convertir a notacion cientifica).

    Returns:
        DataFrame con encabezados normalizados (sin tildes, en mayusculas).

    Raises:
        FileNotFoundError: si la ruta no existe (con la carpeta que si existe).
        ValueError: si la extension no es soportada.
    """
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:12]) or "(vacia)"
        raise FileNotFoundError(
            f"No existe: {p}\n  La carpeta mas cercana que SI existe es: {padre}\n"
            f"  Contiene: {vistos}\n  Corrija la ruta o el nombre del archivo (copielo exacto)."
        )
    dtype = {c: "string" for c in columnas_texto}
    ext = p.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(p, header=fila_encabezado, dtype=dtype, encoding="utf-8")
    elif ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(p, sheet_name=hoja, header=fila_encabezado, dtype=dtype)
    else:
        raise ValueError(f"Extension no soportada: '{ext}'. Use .csv o .xlsx")
    df.columns = [_norm(c) for c in df.columns]
    return df


def exigir_columnas(df: pd.DataFrame, requeridas: dict[str, str], contexto: str) -> pd.DataFrame:
    """Renombra columnas del usuario al contrato NIT/RAZON_SOCIAL/... y valida.

    Args:
        df: base leida.
        requeridas: {nombre_en_su_archivo: nombre_contrato}; los nombres del
            archivo se comparan ya normalizados (sin tildes, mayusculas).
        contexto: etiqueta para mensajes de error (p. ej. "BASE_A").

    Returns:
        DataFrame solo con las columnas del contrato, renombradas.
    """
    mapa = {_norm(k): v for k, v in requeridas.items()}
    faltan = [k for k in mapa if k not in df.columns]
    if faltan:
        raise KeyError(
            f"[{contexto}] No se hallaron las columnas {faltan}.\n"
            f"  Columnas disponibles: {list(df.columns)}\n"
            "  Corrija los nombres en la CONFIGURACION (se comparan sin tildes ni mayusculas)."
        )
    out = df[list(mapa)].rename(columns=mapa).copy()
    for c in out.columns:
        out[c] = out[c].astype("string").fillna("").str.strip()
    return out


# ── Exportacion de vistas (Excel si caben; CSV comprimido si exceden) ──────
# Este notebook NO escribe la carpeta del estandar de salida: el motor
# multicampo (`evaluar_esquema` + `clusters_desde_decisiones`) entrega la
# particion y las decisiones par a par, no el contrato 1.0 completo
# (ID_REGISTRO, ID_ENTIDAD, METODO_UNION…); esa unificacion es de una fase
# posterior. Sus salidas son vistas: CLUSTERS y DECISIONES, escritas con las
# mismas primitivas neutralizadas que usa la libreria para el estandar.


def exportar_vistas(
    tablas: dict[str, pd.DataFrame], carpeta: str | Path, nombre_base: str
) -> list[Path]:
    """Escribe cada vista como .xlsx (formulas neutralizadas) o .csv.gz si no cabe en Excel.

    Returns:
        Rutas de los archivos escritos.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    escritos: list[Path] = []
    for vista, tabla in tablas.items():
        if len(tabla) <= LIMITE_FILAS_EXCEL:
            ruta = carpeta / f"{nombre_base}__{vista}.xlsx"
            escribir_xlsx(tabla, ruta, hoja=vista)
        else:
            ruta = carpeta / f"{nombre_base}__{vista}.csv.gz"
            escribir_csv_gz(tabla, ruta)
        escritos.append(ruta)
        print(f"  Vista escrita: {ruta}")
    return escritos

COLUMNAS_ESQUEMA = ("NIT", "RAZON_SOCIAL", "TELEFONO", "EMAIL", "DIRECCION", "CIUDAD")


def esquema_configurable(
    pesos: dict[str, float],
    corroborar_con: tuple[str, ...] = (),
    umbral_nombre_corroboracion: float = 0.90,
    faltante: PoliticaFaltante = PoliticaFaltante.IGNORAR,
) -> EsquemaCampos:
    """Construye el esquema multicampo con SUS pesos y la corroboracion F3.

    Args:
        pesos: peso por campo; use 0.0 para excluir un campo del score.
        corroborar_con: campos fuertes que pueden levantar el veto de NIT
            (tipicamente ("EMAIL",) o ("EMAIL", "TELEFONO")). Vacio = F3 OFF.
        umbral_nombre_corroboracion: similitud minima de nombre para corroborar.
        faltante: politica ante valores vacios (IGNORAR = neutro, recomendado).
    """
    tipos = {
        "NIT": TipoCampo.IDENTIFICADOR,
        "RAZON_SOCIAL": TipoCampo.NOMBRE_EMPRESA,
        "TELEFONO": TipoCampo.TELEFONO,
        "EMAIL": TipoCampo.EMAIL,
        "DIRECCION": TipoCampo.DIRECCION,
        "CIUDAD": TipoCampo.CIUDAD,
    }
    campos = tuple(
        CampoSpec(col, tipos[col], peso=float(pesos.get(col, 0.0)), faltante=faltante)
        for col in COLUMNAS_ESQUEMA
        if pesos.get(col, 0.0) > 0
    )
    esq = EsquemaCampos(campos=campos, nombre="esquema_notebook_04")
    if corroborar_con:
        esq.corroboracion = CorroboracionVeto(
            campos_corroborantes=tuple(corroborar_con),
            umbral_nombre_empresa=umbral_nombre_corroboracion,
            activa=True,
        )
    return esq


def bloqueo_estandar(umbral_lsh: float = 0.35) -> BloqueoComponible:
    """Bloqueo componible: llaves exactas fuertes + LSH sobre el nombre."""
    return BloqueoComponible(
        [
            LlaveExacta("NIT"),
            LlaveExacta("EMAIL"),
            LlaveExacta("TELEFONO"),
            LSHTexto("RAZON_SOCIAL", umbral=umbral_lsh, permutaciones=64, ngram=3),
        ]
    )


def deduplicar_multicampo(
    df: pd.DataFrame,
    esquema: EsquemaCampos,
    umbral_lsh: float = 0.35,
    max_candidatos: int = 1_000_000,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evalua el esquema y devuelve (df + CLUSTER_ID, decisiones par a par)."""
    faltan = [c for c in {c.nombre for c in esquema.campos} if c not in df.columns]
    if faltan:
        raise KeyError(f"El DataFrame no tiene las columnas del esquema: {faltan}")
    res = evaluar_esquema(df, esquema, bloqueo_estandar(umbral_lsh), max_candidatos=max_candidatos)
    labels = clusters_desde_decisiones(len(df), res.decisiones, respetar_vetos=True)
    salida = df.copy()
    salida["CLUSTER_ID"] = labels
    return salida, res.decisiones


def metricas_contra_verdad(df_cluster: pd.DataFrame, col_verdad: str) -> dict[str, float]:
    """Precision / recall / F1 por pares contra la columna de verdad."""
    m = evaluar_pares(df_cluster[col_verdad].to_numpy(), df_cluster["CLUSTER_ID"].to_numpy())
    return {
        "precision": round(m.precision, 4),
        "recall": round(m.recall, 4),
        "f1": round(m.f1, 4),
        "tp": m.tp,
        "fp": m.fp,
        "fn": m.fn,
        "grupos_verdad": m.n_true_groups,
        "grupos_predichos": m.n_pred_groups,
    }


def demo_sintetica() -> pd.DataFrame:
    """12 registros con verdad conocida; incluye el caso F3 (multi-NIT corroborado)."""
    return pd.DataFrame(
        {
            "REG_ID": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            "VERDAD": [1, 1, 2, 2, 3, 3, 4, 5, 6, 6, 7, 7],
            "NIT": [
                "900111222",
                "900111333",
                "800555666",
                "800555666",
                "",
                "",
                "901234567",
                "901239999",
                "830000111",
                "830000111-4",
                "",
                "",
            ],
            "RAZON_SOCIAL": [
                "COMERCIALIZADORA ANDINA SAS",
                "COMERCIALIZADORA ANDINA S.A.S.",
                "BETA LOGISTICA LTDA",
                "BETA LOGISTICA LIMITADA",
                "CAFE DEL VALLE SA",
                "CAFE DEL VALLE S.A.",
                "DISTRIBUCIONES GAMMA",
                "IMPORTADORA DELTA",
                "EXITO REGIONAL SAS",
                "EXITO REGIONAL S A S",
                "TALLERES OMEGA",
                "TALLERES OMEGA E.U.",
            ],
            "EMAIL": [
                "ventas@andina.co",
                "ventas@andina.co",
                "info@betalog.co",
                "",
                "cafe@valle.co",
                "cafe@valle.co",
                "gamma@gmail.com",
                "gamma@gmail.com",
                "contacto@exito-r.co",
                "contacto@exito-r.co",
                "",
                "",
            ],
            "TELEFONO": [
                "6015550101",
                "6015550101",
                "6045550202",
                "6045550202",
                "",
                "6025550303",
                "3105550404",
                "3105550404",
                "6015550505",
                "6015550505",
                "6075550606",
                "6075550606",
            ],
            "DIRECCION": [
                "CL 100 8A 55",
                "CALLE 100 # 8A-55",
                "CR 50 10 20",
                "CARRERA 50 10-20",
                "AV 6N 25 40",
                "AVENIDA 6 NORTE 25-40",
                "CL 12 3 45",
                "CL 80 20 10",
                "CR 7 32 16",
                "CARRERA 7 # 32-16",
                "CL 5 4 3",
                "CALLE 5 # 4-3",
            ],
            "CIUDAD": [
                "BOGOTA",
                "BOGOTA",
                "MEDELLIN",
                "MEDELLIN",
                "CALI",
                "CALI",
                "BOGOTA",
                "BOGOTA",
                "BOGOTA",
                "BOGOTA",
                "IBAGUE",
                "IBAGUE",
            ],
        }
    )

## 1 · Demo autocontenida — el efecto de la corroboracion F3

Los registros 1 y 2 son la **misma empresa con NITs distintos** (error de digitacion),
pero comparten email exacto y nombre casi identico. Con la corroboracion **OFF** el
veto de NIT los separa; con **ON** (`EMAIL` corrobora) se reunen — sin unir a los
registros 7 y 8, que comparten un gmail generico pero son empresas distintas.

In [ ]:
# ══ EJECUTAR · demo OFF vs ON ═══════════════════════════════════════════════
PESOS_DEMO = {
    "NIT": 3.0,
    "RAZON_SOCIAL": 2.0,
    "TELEFONO": 1.5,
    "EMAIL": 1.5,
    "DIRECCION": 1.0,
    "CIUDAD": 0.5,
}
demo = demo_sintetica()

for etiqueta, corrobora in [("OFF", ()), ("ON  (EMAIL corrobora)", ("EMAIL",))]:
    esq = esquema_configurable(PESOS_DEMO, corroborar_con=corrobora)
    con_cluster, _ = deduplicar_multicampo(demo, esq)
    m = metricas_contra_verdad(con_cluster, "VERDAD")
    unidos = con_cluster.loc[con_cluster["REG_ID"].isin([1, 2]), "CLUSTER_ID"].nunique() == 1
    print(
        f"Corroboracion {etiqueta:<22} P={m['precision']:.4f}  R={m['recall']:.4f}  "
        f"F1={m['f1']:.4f}  | ¿REG 1 y 2 unidos?: {'SI' if unidos else 'NO'}"
    )

con_cluster.sort_values(["CLUSTER_ID", "REG_ID"])[
    ["REG_ID", "VERDAD", "CLUSTER_ID", "NIT", "RAZON_SOCIAL", "EMAIL"]
]

## 2 · Sus datos

Requisitos: un archivo con al menos NIT y razon social; email, telefono, direccion y
ciudad son opcionales (deles peso `0.0` si no existen). Ajuste:

- **PESOS**: cuanto aporta cada campo al score (el NIT ademas **veta** por diseno).
- **CORROBORAR_CON**: `()` = F3 apagada (comportamiento clasico); `("EMAIL",)` o
  `("EMAIL", "TELEFONO")` = NITs distintos pueden unirse solo con corroboracion fuerte.
- **UMBRAL_LSH**: sensibilidad del bloqueo por nombre (0.35 estandar; suba a 0.5 si
  el proceso genera demasiados candidatos en bases muy grandes).
- **MAX_CANDIDATOS**: presupuesto fail-fast de pares; evita agotar la RAM de Colab.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — SUS DATOS: CONFIGURAR Y EJECUTAR
# ══════════════════════════════════════════════════════════════════════════
ARCHIVO = "base_multicampo.xlsx"  # dentro de RUTA_DATOS (Celda A1)
HOJA = 0
MAPA_COLUMNAS = {  # {columna EN SU archivo: campo del esquema}
    "NIT": "NIT",
    "RAZON SOCIAL": "RAZON_SOCIAL",
    "EMAIL": "EMAIL",
    "TELEFONO": "TELEFONO",
    "DIRECCION": "DIRECCION",
    "CIUDAD": "CIUDAD",
}
PESOS = {
    "NIT": 3.0,
    "RAZON_SOCIAL": 2.0,
    "EMAIL": 1.5,
    "TELEFONO": 1.5,
    "DIRECCION": 1.0,
    "CIUDAD": 0.5,
}
CORROBORAR_CON = ("EMAIL", "TELEFONO")  # () para apagar la F3
UMBRAL_LSH = 0.35
MAX_CANDIDATOS = 1_000_000
COLUMNA_VERDAD = "ID_GRUPO_REAL"  # opcional; se conserva si existe

base_cruda = leer_base(RUTA_DATOS / ARCHIVO, hoja=HOJA, columnas_texto=("NIT",))
base = exigir_columnas(base_cruda, MAPA_COLUMNAS, "MULTICAMPO")
col_verdad_entrada = _norm(COLUMNA_VERDAD)
if col_verdad_entrada in base_cruda.columns:
    base[COLUMNA_VERDAD] = base_cruda[col_verdad_entrada].to_numpy()
del base_cruda
for c in COLUMNAS_ESQUEMA:  # completa como vacio lo no mapeado
    if c not in base.columns:
        base[c] = ""
con_cluster, decisiones = deduplicar_multicampo(
    base, esquema_configurable(PESOS, CORROBORAR_CON), UMBRAL_LSH, MAX_CANDIDATOS
)
exportar_vistas({"CLUSTERS": con_cluster, "DECISIONES": decisiones}, RUTA_RESULTADOS, "multicampo")
print(f"{con_cluster['CLUSTER_ID'].nunique():,} empresas unicas de {len(con_cluster):,} registros")

## 3 · Evaluacion contra una verdad de referencia (opcional)

Si su archivo trae una columna con el grupo verdadero (una muestra etiquetada a mano
o un ground truth historico), mida la calidad de la configuracion:

In [ ]:
# ══ EJECUTAR · metricas P/R/F1 (requiere columna de verdad en su archivo) ═══
if COLUMNA_VERDAD in base.columns:
    con_cluster[COLUMNA_VERDAD] = base[COLUMNA_VERDAD].to_numpy()
    print(metricas_contra_verdad(con_cluster, COLUMNA_VERDAD))
else:
    print(
        f"La columna '{COLUMNA_VERDAD}' no esta en el archivo: agregue una verdad "
        "de referencia (aunque sea de una muestra) para calibrar pesos y umbrales."
    )

### Interpretacion y auditoria

- **CLUSTERS** y **DECISIONES** quedan en `RUTA_RESULTADOS` como `multicampo__*.xlsx`
  (o `.csv.gz` si exceden el limite de Excel). Este notebook no escribe la carpeta del
  estandar de salida: el motor multicampo entrega la particion (`CLUSTER_ID`) y las
  decisiones, no el contrato 1.0 completo.
- **DECISIONES**: cada par evaluado con su score por campo, si hubo **veto** de NIT y
  si la corroboracion lo **levanto** (`veto_levantado`). Es el insumo de auditoria: si
  un grupo quedo mal unido, busque ahi el par y el campo responsable.
- **Calibracion sugerida**: parta de los pesos por defecto; si ve falsos positivos por
  nombres genericos, suba el peso del NIT o baje el de RAZON_SOCIAL; si la F3 une de
  mas, quite TELEFONO de `CORROBORAR_CON` o suba `umbral_nombre_corroboracion`.
- Referencia de la Fase 3 y sus salvaguardas: `docs/` del repositorio y el CHANGELOG
  de la version 0.14.0.